# Customer Segmentation & Prediction Project
## Week 11: Advanced Machine Learning Models
This notebook covers customer segmentation using clustering and predictive modeling for distinct segments.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
sns.set(style='whitegrid')

### 1. Load and Inspect Data

In [ ]:
df = pd.read_csv('segmentation_data.csv')
print(df.head())
print(df.info())
print(df.describe())

### 2. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Churn')
plt.title('Churn Distribution')
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', kde=True)
plt.title('Monthly Charges by Churn')
plt.show()

### 3. Preprocessing for Clustering
We will encode categorical variables and scale numerical ones.

In [ ]:
# Encoding
le = LabelEncoder()
cat_cols = ['Contract', 'PaymentMethod', 'PaperlessBilling']
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

# Scaling
scaler = StandardScaler()
num_cols = ['Tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
X_scaled = scaler.fit_transform(df[num_cols])

### 4. Clustering: K-Means & Optimal Clusters

In [ ]:
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.show()

Applying K-Means with 3 clusters (based on Elbow method).

In [ ]:
kmeans = KMeans(n_clusters=3, init='k-means++', random_state=42)
df['Cluster'] = kmeans.fit_transform(X_scaled).argmin(axis=1) # Simplified assignment
df['Cluster'] = kmeans.labels_
print(df['Cluster'].value_counts())

### 5. Advanced Clustering: DBSCAN

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)
print('DBSCAN Cluster Labels:', np.unique(dbscan_labels))

### 6. Segment Profiling & Analysis

In [ ]:
cluster_profiles = df.groupby('Cluster')[num_cols + ['Churn']].mean()
print(cluster_profiles)

# Assigning names based on characteristics
def name_segments(cluster):
    if cluster == 0: return 'High-Value Loyalists'
    elif cluster == 1: return 'Budget-Conscious Newbies'
    else: return 'Premium Spenders'

df['SegmentName'] = df['Cluster'].apply(name_segments)
print(df['SegmentName'].value_counts())

### 7. Prediction Models for each Segment
We will build a Random Forest model for each segment to predict Churn.

In [ ]:
segments = df['SegmentName'].unique()
evaluation_results = []

for segment in segments:
    print(f'\nProcessing Segment: {segment}')
    seg_df = df[df['SegmentName'] == segment]
    X = seg_df.drop(['CustomerID', 'Churn', 'SegmentName', 'Cluster'], axis=1)
    y = seg_df['Churn']
    
    if len(y.unique()) < 2:
        print(f'Skipping {segment} due to single class in target.')
        continue
        
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    rf = RandomForestClassifier(random_state=42)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
    
    print(f'Accuracy: {acc:.2f}, F1: {f1:.2f}, ROC-AUC: {roc:.2f}')
    
    evaluation_results.append({
        'Segment': segment,
        'Accuracy': acc,
        'F1-Score': f1,
        'ROC-AUC': roc
    })

### 8. Hyperparameter Tuning using Grid Search

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='f1')
# Let's tune for the overall dataset for simplicity in demonstration
X_all = df.drop(['CustomerID', 'Churn', 'SegmentName', 'Cluster'], axis=1)
y_all = df['Churn']
grid_search.fit(X_all, y_all)
print('Best Params:', grid_search.best_params_)
print('Best F1 Score:', grid_search.best_score_)

### 9. Export Results

In [ ]:
eval_df = pd.DataFrame(evaluation_results)
eval_df.to_csv('model_evaluation_results.csv', index=False)
print('Evaluation results saved.')